In [0]:
spark.sql("""
CREATE TABLE fraud_catalog.audit.pipeline_audit (
    pipeline_name STRING,
    run_time TIMESTAMP,
    total_records BIGINT,
    fraud_count BIGINT,
    status STRING,
    error_message STRING)
USING DELTA
LOCATION 'abfss://fraudexternal@fraudstorageacc1234.dfs.core.windows.net/audit/pipeline_audit'
""")

DataFrame[]

In [0]:
import logging

logging.basicConfig(level=logging.INFO)

try:
    logging.info("Audit started")

    # read Gold table
    df = spark.table("fraud_catalog.gold.fraud_transactions")

    total_records = df.count()
    fraud_count = df.filter("fraud_flag = 1").count()

    logging.info(f"Total records: {total_records}")
    logging.info(f"Fraud records: {fraud_count}")

    # insert audit record
    spark.sql(f"""INSERT INTO fraud_catalog.audit.pipeline_audit
        VALUES ('fraud_pipeline',current_timestamp(),{total_records},{fraud_count},'SUCCESS',NULL)""")

    logging.info("Audit completed")

except Exception as e:
    logging.error(str(e))

    error_msg = str(e).replace("'", " ")

    spark.sql(f"""
        INSERT INTO fraud_catalog.audit.pipeline_audit
        VALUES ('fraud_pipeline',current_timestamp(),0,0,'FAIL','{error_msg}')""")
    raise

INFO:root:Audit started
INFO:root:Total records: 1048575
INFO:root:Fraud records: 5203
INFO:root:Audit completed


In [0]:
display(spark.table("fraud_catalog.audit.pipeline_audit"))

pipeline_name,run_time,total_records,fraud_count,status,error_message
fraud_pipeline,2026-04-06T14:00:32.535272Z,1048575,5203,SUCCESS,null
fraud_pipeline,2026-04-05T15:59:59.079392Z,1048575,5203,SUCCESS,null
fraud_pipeline,2026-04-06T14:27:52.506129Z,1048575,5203,SUCCESS,null
